# Auriga — LLM Model Download & Verification

This notebook downloads both LLM models required by `MindEngine` and verifies their SHA-256 checksums.

**After running all cells**, download `auriga_models.zip` from the Files panel (left sidebar → 📁) and extract it into your local `AURIGA/app/src/main/assets/` directory.

---

| Model | Size | Licence | Use |
|---|---|---|---|
| `qwen2_5_0_5b_q8.bin` | ~519 MB | Apache-2.0 | Fallback / low-RAM devices |
| `gemma2b_q4.bin` | ~2.52 GB | Gemma (HF login required) | Primary / flagship devices |

> **Gemma prerequisite:** accept the licence at [litert-community/Gemma2-2B-IT](https://huggingface.co/litert-community/Gemma2-2B-IT) before running the Gemma cell.

In [ ]:
# Install required library
!pip install -q huggingface_hub

In [ ]:
import hashlib
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

OUT_DIR = Path('/content/auriga_assets')
OUT_DIR.mkdir(exist_ok=True)

EXPECTED = {
    'qwen2_5_0_5b_q8.bin':  '54806eb754fe80fe6ed42d055ea56099ae0a273a52bda6437290cc00c501000b',
    'gemma2b_q4.bin':        '29ff136fd298e611296e10e9b511c86f42d1291b5b8bfc18c42178e733b679a9',
}

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            block = f.read(chunk)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def verify(dest_name):
    path = OUT_DIR / dest_name
    if not path.exists():
        print(f'  ✗  {dest_name}: file not found')
        return False
    actual = sha256(path)
    expected = EXPECTED[dest_name]
    size_mb = path.stat().st_size / 1024 / 1024
    if actual == expected:
        print(f'  ✓  {dest_name}  ({size_mb:.0f} MB)  SHA-256 OK')
        return True
    else:
        print(f'  ✗  {dest_name}: SHA-256 MISMATCH')
        print(f'     expected: {expected}')
        print(f'     actual:   {actual}')
        return False

print('Setup complete.')

## Step 1 — Download Qwen 2.5 0.5B (no login needed)

In [ ]:
print('Downloading Qwen 2.5 0.5B (~519 MB)...')
path = hf_hub_download(
    repo_id='litert-community/Qwen2.5-0.5B-Instruct',
    filename='Qwen2.5-0.5B-Instruct_multi-prefill-seq_q8_ekv1280.tflite',
    local_dir=str(OUT_DIR),
    local_dir_use_symlinks=False,
)
# Rename to the filename MindEngine expects
dest = OUT_DIR / 'qwen2_5_0_5b_q8.bin'
Path(path).rename(dest)
print('Download complete. Verifying...')
verify('qwen2_5_0_5b_q8.bin')

## Step 2 — Download Gemma 2 2B IT (HF login required)

**Before running this cell:**
1. Accept the Gemma licence at [litert-community/Gemma2-2B-IT](https://huggingface.co/litert-community/Gemma2-2B-IT)
2. Create a read-access token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Paste the token when prompted below

In [ ]:
from huggingface_hub import login
login()  # prompts for your HF token — never stored in the notebook

In [ ]:
print('Downloading Gemma 2 2B IT (~2.52 GB)...')
path = hf_hub_download(
    repo_id='litert-community/Gemma2-2B-IT',
    filename='Gemma2-2B-IT_multi-prefill-seq_q8_ekv1280.tflite',
    local_dir=str(OUT_DIR),
    local_dir_use_symlinks=False,
)
dest = OUT_DIR / 'gemma2b_q4.bin'
Path(path).rename(dest)
print('Download complete. Verifying...')
verify('gemma2b_q4.bin')

## Step 3 — Package and download

In [ ]:
import zipfile

zip_path = '/content/auriga_models.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_STORED) as zf:
    for f in OUT_DIR.iterdir():
        if f.suffix == '.bin':
            zf.write(f, f.name)
            print(f'  Added: {f.name}  ({f.stat().st_size / 1024 / 1024:.0f} MB)')

print(f'\nZip created: {zip_path}')
print('Download it from the Files panel (left sidebar) and extract into:')
print('  AURIGA/app/src/main/assets/')

## After downloading the zip

```bash
unzip auriga_models.zip -d AURIGA/app/src/main/assets/
```

Then build the APK in Android Studio:
- Flavour: `navi`
- **Build → Generate Signed APK**

MindEngine loads Gemma on devices with ≥ 3,500 MB RAM and Qwen on everything else — no configuration needed.